<a href="https://colab.research.google.com/github/Hanaa879/Student-Performance-Analytics-System/blob/main/Student_Performance_Analytics_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title  Student Performance Analytics System & Dashboard { run: "auto" }

# We have imported these python libraries to help us analyse the data, train the model and produce a good output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder



In [2]:
#we will be connecting google drive to the Colab using code below
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = pd.read_csv('/content/drive/MyDrive/archive (2)/StudentsPerformance.csv')


In [4]:
# Display the first 5 rows to understand the structure
data.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [5]:
#So since all the ml models are in one cell we are giving a general cell to pre process the data but inside each ml model we need to go into specifics only in those sections
# Create Targets for both Classification and Regression
data['Average_Score'] = data[['math score', 'reading score', 'writing score']].mean(axis=1)
data['Performance'] = np.where(data['Average_Score'] >= 60, 1, 0)

# Target for Linear Regression (Continuous numeric marks)
y_reg = data['Average_Score']

# Target for Logistic Regression, KNN, Decision Tree, Random Forest (Binary Pass/Fail)
y = data['Performance']

# Separate features and convert text categories properly using One-Hot Encoding
# This avoids treating categorical data as arbitrary sequential numbers
X_raw = data.drop(['math score', 'reading score', 'writing score', 'Average_Score', 'Performance'], axis=1)
X = pd.get_dummies(X_raw, drop_first=True)

# 3. Scale features for distance-based models (KNN, K-Means, PCA)
# This prevents columns with wider ranges from dominating the algorithms
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Preview the ready-to-use features
print("--- Prepared Features Preview (X.head()) ---")
print(X.head())
print("-" * 44)

--- Prepared Features Preview (X.head()) ---
   gender_male  race/ethnicity_group B  race/ethnicity_group C  \
0        False                    True                   False   
1        False                   False                    True   
2        False                    True                   False   
3         True                   False                   False   
4         True                   False                    True   

   race/ethnicity_group D  race/ethnicity_group E  \
0                   False                   False   
1                   False                   False   
2                   False                   False   
3                   False                   False   
4                   False                   False   

   parental level of education_bachelor's degree  \
0                                           True   
1                                          False   
2                                          False   
3                              

In [6]:
#@markdown Select a model from the dropdown menu to update the dashboard view.
# 2. The code below helps us select a dashboard to display in google collabs without having to download
Select_Model = "KNN" #@param ["Linear Regression", "Logistic Regression", "KNN", "Decision Tree", "Random Forest", "K-Means", "PCA"]

# 3. DASHBOARD code
if Select_Model == "Decision Tree":
    print("Decision Tree → Academic performance analysis  ")

    # 1. Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Testing samples: {X_test.shape[0]}")

    # 2. Train Decision Tree
    model = DecisionTreeClassifier(max_depth=3, random_state=42)
    model.fit(X_train, y_train)
    print("Model trained successfully!")

    # 3. Evaluate and predict
    y_pred = model.predict(X_test)
    print("Accuracy Score:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Fail', 'Pass']))

    # 4. Plot tree visual
    plt.figure(figsize=(20, 10))
    plot_tree(model, feature_names=X.columns, class_names=['Fail', 'Pass'], filled=True, rounded=True, fontsize=12)
    plt.show()

    # 5. Extract Feature Importance (This belongs ONLY to Decision Tree)
    importances = model.feature_importances_
    feature_imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances}).sort_values(by='Importance', ascending=False)
    print("\nKey factors influencing academic performance:")
    print(feature_imp_df)

elif Select_Model == "Linear Regression":
    print("  Linear Regression → Marks prediction  ")
    # Your Linear Regression code goes here

elif Select_Model == "Logistic Regression":
    print(" Logistic Regression → Pass/Fail prediction  ")
    # Your Logistic Regression code goes here

elif Select_Model == "KNN":
    print(" KNN → Similar student retrieval ")

    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.neighbors import NearestNeighbors

    # 1. Prepare features using your snippet
    X = data.copy()

    categorical_cols = ['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course']
    numerical_cols = ['math score', 'reading score', 'writing score']

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(), categorical_cols)
        ])

    X_processed = preprocessor.fit_transform(X)
    print("Data preprocessed successfully! Shape:", X_processed.shape)

    # 2. Initialize and Train KNN
    knn = NearestNeighbors(n_neighbors=6, metric='euclidean')
    knn.fit(X_processed)
    print("KNN Model is ready!")

    # 3. Use the model to find similar students
    distances, indices = knn.kneighbors(X_processed[0].reshape(1, -1))
    print("\nIndices of 6 most similar students:", indices[0])
    print("Distances to these students:", distances[0])

    # 4. Define the retrieval function inside the KNN block
    def find_similar_students(student_id, dataset, processed_data, model):
        # Get the processed row for the target student
        target_student = processed_data[student_id].reshape(1, -1)

        # Find the distances and indices of the nearest neighbors
        distances, indices = model.kneighbors(target_student)

        print(f"--- Target Student (ID: {student_id}) ---")
        print(dataset.iloc[student_id].to_frame().T)
        print("\n" + "="*50 + "\n")
        print(f"--- Top 5 Most Similar Students ---")

        # Skip the first index because it's the target student themselves
        similar_indices = indices[0][1:]
        similar_distances = distances[0][1:]

        # Display results with match distance
        results = dataset.iloc[similar_indices].copy()
        results.insert(0, 'Similarity Distance', similar_distances)
        return results

    # 5. Call the function (Corrected indentation: now runs after the function definition)
    sim_results = find_similar_students(student_id=42, dataset=data, processed_data=X_processed, model=knn)
    display(sim_results)

elif Select_Model == "Random Forest":
    print(" Random Forest → Dropout prediction risk ")
    # Your Random Forest code goes here

elif Select_Model == "K-Means":
    print(" K-Means → Student grouping cluster  ")
    from sklearn.cluster import KMeans

    # K-Means strictly groups the scaled data into distinct clusters
    kmeans_model = KMeans(n_clusters=3, random_state=42)
    clusters = kmeans_model.fit_predict(X_scaled)

    print("K-Means clustering completed successfully!")
    # (Optional: Add cluster count summaries here later)

elif Select_Model == "PCA":
    print(" PCA → Feature reduction  ")
    # Your PCA code goes here

 KNN → Similar student retrieval 
Data preprocessed successfully! Shape: (1000, 20)
KNN Model is ready!

Indices of 6 most similar students: [  0 355 969 232 765 350]
Distances to these students: [0.         0.77211406 0.9335165  1.17489417 1.42644842 1.44220649]
--- Target Student (ID: 42) ---
    gender race/ethnicity parental level of education     lunch  \
42  female        group B          associate's degree  standard   

   test preparation course math score reading score writing score  \
42                    none         53            58            65   

   Average_Score Performance  
42     58.666667           0  


--- Top 5 Most Similar Students ---


,Similarity Distance,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score,Average_Score,Performance
609,0.475647,female,group B,associate's degree,standard,none,58,63,65,62.000000,1
203,0.822725,female,group B,associate's degree,standard,none,57,69,68,64.666667,1
484,0.873644,female,group B,associate's degree,standard,none,49,52,54,51.666667,0
80,1.229833,female,group B,associate's degree,standard,none,47,49,50,48.666667,0
552,1.476738,female,group B,associate's degree,standard,none,40,48,50,46.000000,0
